# Bacterial Assembly Results


In [ ]:
# import necessary modules
import os
import glob
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["savefig.dpi"]=300


In [ ]:
def as_list(x):
	if isinstance(x, str):
		return([x])
	return(list(x))

input_checkm=as_list(snakemake.input.checkm)
input_sourmash=as_list(snakemake.input.sourmash)
input_gtdbtk=as_list(snakemake.input.gtdbtk)
input_quast=snakemake.input.quast

SAMPLING=snakemake.params.sampling

output_summary_html=snakemake.output.summary_html
output_summary_csv=snakemake.output.summary_csv
output_checkm_png=snakemake.output.checkm_png
output_checkm_svg=snakemake.output.checkm_svg
output_taxonomy_png=snakemake.output.taxonomy_png
output_taxonomy_svg=snakemake.output.taxonomy_svg
output_quality_taxonomy_png=snakemake.output.quality_taxonomy_png
output_quality_taxonomy_svg=snakemake.output.quality_taxonomy_svg


## Parse CheckM


In [ ]:
def parse_checkm_dir(path):
	sample=os.path.basename(path).replace("_checkM_" + SAMPLING, "")
	candidates=[]
	candidates.extend(glob.glob(path + "/*storage/bin_stats_ext.tsv"))
	candidates.extend(glob.glob(path + "/storage/bin_stats_ext.tsv"))
	candidates.extend(glob.glob(path + "/*quality_report.tsv"))
	candidates.extend(glob.glob(path + "/quality_report.tsv"))
	candidates.extend(glob.glob(path + "/*lineage.ms"))
	candidates.extend(glob.glob(path + "/*.tsv"))
	out={"sample":sample, "checkm_completeness":np.nan, "checkm_contamination":np.nan, "checkm_strain_heterogeneity":np.nan, "checkm_marker_lineage":np.nan, "checkm_genome_size":np.nan}
	for candidate in candidates:
		try:
			df=pd.read_csv(candidate, sep="\t")
			cols=list(df.columns)
			if "Completeness" in cols and "Contamination" in cols:
				row=df.iloc[0]
				out["checkm_completeness"]=row.get("Completeness", np.nan)
				out["checkm_contamination"]=row.get("Contamination", np.nan)
				out["checkm_strain_heterogeneity"]=row.get("Strain heterogeneity", np.nan)
				out["checkm_marker_lineage"]=row.get("Marker lineage", np.nan)
				out["checkm_genome_size"]=row.get("Genome size", np.nan)
				return out
		except Exception:
			pass
	for candidate in candidates:
		try:
			with open(candidate) as handle:
				for line in handle:
					if "Completeness" in line or "completeness" in line:
						pass
		except Exception:
			pass
	return out

checkm_rows=[]
for path in input_checkm:
	checkm_rows.append(parse_checkm_dir(path))

checkm_df=pd.DataFrame(checkm_rows)
for col in ["checkm_completeness", "checkm_contamination", "checkm_strain_heterogeneity", "checkm_genome_size"]:
	if col in checkm_df.columns:
		checkm_df[col]=pd.to_numeric(checkm_df[col], errors="coerce")
checkm_df


## Parse Sourmash classifications


In [ ]:
def parse_sourmash(path):
	sample=os.path.basename(path)
	sample=sample.replace("_" + SAMPLING + ".classifications.csv", "")
	sample=sample.replace("_pacbio_" + SAMPLING + ".classifications.csv", "")
	sample=sample.replace("_pacbio_only_" + SAMPLING + ".classifications.csv", "")
	sample=sample.replace("_pacbio_hybrid_" + SAMPLING + ".classifications.csv", "")
	out={"sample":sample, "sourmash_taxonomy":np.nan, "sourmash_superkingdom":np.nan, "sourmash_phylum":np.nan, "sourmash_class":np.nan, "sourmash_order":np.nan, "sourmash_family":np.nan, "sourmash_genus":np.nan, "sourmash_species":np.nan}
	try:
		df=pd.read_csv(path)
		if len(df)==0:
			return out
		if "lineage" in df.columns:
			out["sourmash_taxonomy"]=df.iloc[0]["lineage"]
		if "rank" in df.columns and "name" in df.columns:
			for rank in ["superkingdom", "phylum", "class", "order", "family", "genus", "species"]:
				sub=df[df["rank"]==rank]
				if len(sub)>0:
					out["sourmash_" + rank]=sub.iloc[0]["name"]
		for col in df.columns:
			lcol=col.lower()
			if lcol in ["superkingdom", "phylum", "class", "order", "family", "genus", "species"]:
				out["sourmash_" + lcol]=df.iloc[0][col]
	except Exception:
		pass
	return out

sourmash_rows=[]
for path in input_sourmash:
	sourmash_rows.append(parse_sourmash(path))

sourmash_df=pd.DataFrame(sourmash_rows)
sourmash_df


## Parse GTDB-Tk


In [ ]:
def parse_gtdbtk_dir(path):
	rows=[]
	files=[]
	files.extend(glob.glob(path + "/classify/gtdbtk.bac120.summary.tsv"))
	files.extend(glob.glob(path + "/gtdbtk.bac120.summary.tsv"))
	files.extend(glob.glob(path + "/**/gtdbtk.bac120.summary.tsv", recursive=True))
	for file in files:
		try:
			df=pd.read_csv(file, sep="\t")
			if "user_genome" in df.columns:
				for _, row in df.iterrows():
					sample=str(row["user_genome"])
					sample=sample.replace(".fasta", "")
					rows.append({
						"sample":sample,
						"gtdbtk_classification":row.get("classification", np.nan),
						"gtdbtk_fastani_reference":row.get("fastani_reference", np.nan),
						"gtdbtk_fastani_ani":row.get("fastani_ani", np.nan),
						"gtdbtk_fastani_af":row.get("fastani_af", np.nan),
						"gtdbtk_note":row.get("note", np.nan)
					})
		except Exception:
			pass
	return rows

gtdbtk_rows=[]
for path in input_gtdbtk:
	gtdbtk_rows.extend(parse_gtdbtk_dir(path))

gtdbtk_df=pd.DataFrame(gtdbtk_rows)
if len(gtdbtk_df)==0:
	gtdbtk_df=pd.DataFrame(columns=["sample", "gtdbtk_classification", "gtdbtk_fastani_reference", "gtdbtk_fastani_ani", "gtdbtk_fastani_af", "gtdbtk_note"])
for col in ["gtdbtk_fastani_ani", "gtdbtk_fastani_af"]:
	if col in gtdbtk_df.columns:
		gtdbtk_df[col]=pd.to_numeric(gtdbtk_df[col], errors="coerce")
gtdbtk_df


# Parse quast

In [ ]:
quast_df=pd.read_csv(input_quast, sep="\t")

if "Assembly" not in quast_df.columns:
	quast_df=quast_df.rename(columns={quast_df.columns[0]: "Assembly"})

quast_df["sample"]=quast_df["Assembly"].astype(str)
quast_df["sample"]=quast_df["sample"].str.replace("_spades_filtered_scaffolds." + SAMPLING, "", regex=False)
quast_df["sample"]=quast_df["sample"].str.replace("_contigs_hifiasm." + SAMPLING, "", regex=False)
quast_df["sample"]=quast_df["sample"].str.replace("_contigs_flye." + SAMPLING, "", regex=False)
quast_df["sample"]=quast_df["sample"].str.replace("racon_", "", regex=False)
quast_df["sample"]=quast_df["sample"].str.replace("_contigs_2_flye." + SAMPLING, "", regex=False)
quast_df["sample"]=quast_df["sample"].str.replace("polypolish_", "", regex=False)

quast_cols=["sample", "# contigs (>= 1000 bp)", "Total length", "N50", "Largest contig", "GC (%)"]
quast_cols=[col for col in quast_cols if col in quast_df.columns]

quast_df=quast_df[quast_cols].copy()
quast_df=quast_df.rename(columns={
	"# contigs (>= 1000 bp)": "assembly_contigs_1000bp",
	"Total length": "assembly_total_length",
	"N50": "assembly_N50",
	"Largest contig": "assembly_largest_contig",
	"GC (%)": "assembly_GC"
})

## Merge bacterial result tables


In [ ]:
summary_df=checkm_df.merge(quast_df, on="sample", how="outer").merge(sourmash_df, on="sample", how="outer").merge(gtdbtk_df, on="sample", how="outer")

def checkm_quality(row):
	if pd.isna(row.get("checkm_completeness", np.nan)) or pd.isna(row.get("checkm_contamination", np.nan)):
		return "Not available"
	if row["checkm_completeness"] >= 90 and row["checkm_contamination"] <= 5:
		return "High-quality"
	if row["checkm_completeness"] >= 50 and row["checkm_contamination"] <= 10:
		return "Medium-quality"
	return "Low-quality"

summary_df["checkm_quality"]=summary_df.apply(checkm_quality, axis=1)

if "gtdbtk_classification" in summary_df.columns:
	summary_df["gtdbtk_genus"]=summary_df["gtdbtk_classification"].astype(str).str.extract(r"g__([^;]*)")[0]
	summary_df["gtdbtk_species"]=summary_df["gtdbtk_classification"].astype(str).str.extract(r"s__([^;]*)")[0]

summary_df=summary_df.sort_values("sample")
summary_df.to_csv(output_summary_csv, index=False)

summary_df_out=summary_df.style.background_gradient(cmap="RdYlGn", subset=[col for col in ["assembly_contigs_1000bp", "assembly_total_length", "assembly_N50", "assembly_largest_contig", "assembly_GC", "checkm_completeness", "checkm_contamination", "gtdbtk_fastani_ani", "gtdbtk_fastani_af"] if col in summary_df.columns]).to_html()
with open(output_summary_html, "w") as fp:
	fp.write(summary_df_out)

summary_df


## CheckM completeness and contamination


In [ ]:
plot_df=summary_df.dropna(subset=["checkm_completeness", "checkm_contamination"]).copy()
fig_width=max(8, 0.45*len(plot_df))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=plot_df.melt(id_vars=["sample", "checkm_quality"], value_vars=["checkm_completeness", "checkm_contamination"], var_name="metric", value_name="percent")
plot_df["metric"]=plot_df["metric"].replace({"checkm_completeness":"Completeness", "checkm_contamination":"Contamination"})

sns.barplot(data=plot_df, x="sample", y="percent", hue="metric", ax=ax)
ax.axhline(90, linestyle="--", linewidth=1, alpha=0.5)
ax.axhline(5, linestyle=":", linewidth=1, alpha=0.5)
ax.set_xlabel("Sample")
ax.set_ylabel("Percent")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="CheckM metric")

fig.savefig(output_checkm_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_checkm_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## Taxonomic overview


In [ ]:
tax_col="gtdbtk_genus"
if tax_col not in summary_df.columns or summary_df[tax_col].replace("nan", np.nan).dropna().empty:
	tax_col="sourmash_genus"

plot_df=summary_df.copy()
if tax_col in plot_df.columns:
	plot_df[tax_col]=plot_df[tax_col].replace("", np.nan).fillna("Unclassified")
	tax_counts=plot_df.groupby(tax_col).size().reset_index(name="count").sort_values("count", ascending=False)
	fig_width=max(8, 0.7*len(tax_counts))
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.barplot(data=tax_counts, x=tax_col, y="count", ax=ax)
	ax.set_xlabel(tax_col)
	ax.set_ylabel("Number of samples")
	ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
	fig.savefig(output_taxonomy_png, format="png", bbox_inches="tight", transparent=True)
	fig.savefig(output_taxonomy_svg, format="svg", bbox_inches="tight", transparent=True)
	plt.show()
else:
	print("No genus-level taxonomy column available.")


## Quality by taxonomy


In [ ]:
tax_col="gtdbtk_genus"
if tax_col not in summary_df.columns or summary_df[tax_col].replace("nan", np.nan).dropna().empty:
	tax_col="sourmash_genus"

plot_df=summary_df.copy()
if tax_col in plot_df.columns:
	plot_df[tax_col]=plot_df[tax_col].replace("", np.nan).fillna("Unclassified")
	fig_width=max(8, 0.7*plot_df[tax_col].nunique())
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.scatterplot(data=plot_df, x="checkm_completeness", y="checkm_contamination", hue=tax_col, style="checkm_quality", s=100, ax=ax)
	ax.axvline(90, linestyle="--", linewidth=1, alpha=0.5)
	ax.axhline(5, linestyle="--", linewidth=1, alpha=0.5)
	ax.set_xlabel("CheckM completeness (%)")
	ax.set_ylabel("CheckM contamination (%)")
	ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Taxonomy")
	fig.savefig(output_quality_taxonomy_png, format="png", bbox_inches="tight", transparent=True)
	fig.savefig(output_quality_taxonomy_svg, format="svg", bbox_inches="tight", transparent=True)
	plt.show()
else:
	print("No taxonomy column available.")
